In [11]:
# ============================================================
# INITIALIZE RETRIEVER FOR DIAGNOSTIC TEST
# ============================================================

from pathlib import Path
import sys

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter


# ============================================================
# PATHS
# ============================================================

CURRENT_DIR = Path.cwd()

# rag/
RAG_DIR = CURRENT_DIR.parent

# rag/embeddings/processed/chunks/chunks.json
CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ============================================================
# CHECK CHUNKS FILE
# ============================================================

print("=" * 70)
print("DIAGNOSTIC INITIALIZATION")
print("=" * 70)

print("\nChunks file:")
print(CHUNKS_FILE)

print("\nChunks file exists:")
print(CHUNKS_FILE.exists())


# ============================================================
# INITIALIZE COMPONENTS
# ============================================================

hybrid_retriever = HybridRetriever()

reranker = CrossEncoderReranker()

metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("\nAll components initialized successfully.")

DIAGNOSTIC INITIALIZATION

Chunks file:
C:\Users\User\Nasa Rag SYS\rag\embeddings\processed\chunks\chunks.json

Chunks file exists:
True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.
Loaded 1200 chunks.

All components initialized successfully.


In [12]:
# Test what HybridRetriever actually returns

test_result = hybrid_retriever.search(
    query="How does oxygen concentration affect flame behavior in microgravity?",
    dense_top_k=20,
    bm25_top_k=20,
    top_k=5
)

print("=" * 70)
print("HYBRID RESULT STRUCTURE")
print("=" * 70)

print(test_result[0].keys())

print("\nFirst result:")
print(test_result[0])

HYBRID RESULT STRUCTURE
dict_keys(['chunk_id', 'rrf_score', 'dense_rank', 'bm25_rank', 'dense_score', 'bm25_score', 'hybrid_rank'])

First result:
{'chunk_id': 243, 'rrf_score': 0.029726775956284153, 'dense_rank': 1, 'bm25_rank': 15, 'dense_score': 0.7074596285820007, 'bm25_score': 2.2567201934900503, 'hybrid_rank': 1}


In [13]:
# ============================================================
# TEST ENRICHMENT
# ============================================================

enriched_results = metadata_filter.enrich_results(
    test_result
)

print("=" * 70)
print("ENRICHED RESULT STRUCTURE")
print("=" * 70)

print("\nKeys:")
print(enriched_results[0].keys())

print("\nMetadata:")
print(
    enriched_results[0].get(
        "metadata",
        {}
    )
)

print("\nText:")
print(
    enriched_results[0].get(
        "text",
        ""
    )[:1000]
)

ENRICHED RESULT STRUCTURE

Keys:
dict_keys(['chunk_id', 'rrf_score', 'dense_rank', 'bm25_rank', 'dense_score', 'bm25_score', 'hybrid_rank', 'text', 'metadata'])

Metadata:
{'experiment_id': 'EXP-0061', 'source_investigation': 'BASS-II', 'material': 'Nomex', 'material_type': 'solid', 'flow_configuration': 'opposed', 'oxygen_concentration_pct': 15.27, 'pressure_kpa': 68.6, 'flow_velocity_cm_s': 25.84, 'ambient_temperature_c': 23.7, 'outcome': 'self_extinguished', 'risk_level': 'medium', 'test_date': '2026-02-18', 'chunk_type': 'risk_summary', 'source': 'EXP-0061.md'}

Text:
# Experiment EXP-0061

## Fire Risk

- **Risk Score:** 24.7
- **Risk Level:** medium

## Experimental Summary

Experiment EXP-0061 was conducted as part of the
BASS-II investigation using Nomex.

The experiment used an oxygen concentration of
15.27%, a pressure of
68.6 kPa, and a flow velocity of
25.84 cm/s.

The observed flame spread rate was
1.299 mm/s, with a heat release rate of
1.258 kW.

The observed outcome was

In [14]:
# ============================================================
# TEST 30 QUESTIONS
# HYBRID + ENRICHMENT + CROSS-ENCODER
# ============================================================

def test_30_questions():

    TEST_QUERIES = [

        # ====================================================
        # OXYGEN
        # ====================================================

        "How does oxygen concentration affect flame behavior in microgravity?",
        "What happens to flame spread when oxygen concentration changes?",
        "How does oxygen concentration influence heat release rate?",

        # ====================================================
        # PRESSURE
        # ====================================================

        "How does pressure affect flame behavior?",
        "What is the effect of pressure on flame spread?",

        # ====================================================
        # FLOW VELOCITY
        # ====================================================

        "How does flow velocity affect flame spread?",
        "What happens to the flame when flow velocity increases?",

        # ====================================================
        # TEMPERATURE
        # ====================================================

        "How does ambient temperature affect combustion?",
        "What is the relationship between temperature and flame behavior?",

        # ====================================================
        # IGNITION
        # ====================================================

        "What factors affect ignition time?",
        "How quickly do different materials ignite?",

        # ====================================================
        # FLAME SPREAD
        # ====================================================

        "What factors influence flame spread rate?",
        "Which experiments had the highest flame spread rate?",

        # ====================================================
        # HEAT RELEASE
        # ====================================================

        "What factors affect heat release rate?",
        "Which experiments produced high heat release rates?",

        # ====================================================
        # SOOT
        # ====================================================

        "How does combustion produce soot in microgravity?",
        "Which experiments had high soot volume fractions?",

        # ====================================================
        # MATERIALS
        # ====================================================

        "Which materials were tested in the experiments?",
        "How does material type affect flame behavior?",
        "Which materials showed self-extinguishing behavior?",

        # ====================================================
        # OUTCOME
        # ====================================================

        "Which experiments resulted in flame extinction?",
        "Which experiments resulted in self-extinguishment?",

        # ====================================================
        # FIRE RISK
        # ====================================================

        "What conditions are associated with high fire risk?",
        "Which experiments had the highest risk scores?",
        "How is fire risk related to combustion conditions?",

        # ====================================================
        # EXPERIMENTAL CONDITIONS
        # ====================================================

        "How do experimental conditions affect flame behavior?",
        "What conditions were used in the microgravity experiments?",

        # ====================================================
        # COMBINED
        # ====================================================

        "How do oxygen concentration and flow velocity affect flame spread?",
        "How are oxygen concentration, heat release, and flame spread related?",
        "What experimental conditions lead to flame extinction?"
    ]

    print("=" * 90)
    print("30-QUESTION RETRIEVAL TEST")
    print("=" * 90)

    print(
        f"\nTotal questions: {len(TEST_QUERIES)}"
    )

    # ========================================================
    # COMPONENTS
    # ========================================================

    hybrid_retriever = HybridRetriever()

    reranker = CrossEncoderReranker()

    metadata_filter = MetadataFilter(
        chunks_file=str(CHUNKS_FILE)
    )

    # ========================================================
    # RESULTS
    # ========================================================

    all_results = []

    # ========================================================
    # RUN QUESTIONS
    # ========================================================

    for question_number, query in enumerate(
        TEST_QUERIES,
        start=1
    ):

        print("\n\n")

        print("=" * 90)

        print(
            f"QUESTION {question_number}/"
            f"{len(TEST_QUERIES)}"
        )

        print("=" * 90)

        print("\nQUERY:")

        print(query)

        # ====================================================
        # HYBRID
        # ====================================================

        hybrid_results = hybrid_retriever.search(
            query=query,
            dense_top_k=20,
            bm25_top_k=20,
            top_k=20
        )

        # ====================================================
        # ENRICH
        # ====================================================

        enriched_results = (
            metadata_filter.enrich_results(
                hybrid_results
            )
        )

        # ====================================================
        # HYBRID TOP 5
        # ====================================================

        print("\n" + "-" * 90)
        print("HYBRID TOP 5")
        print("-" * 90)

        for result in enriched_results[:5]:

            metadata = result.get(
                "metadata",
                {}
            )

            experiment_id = metadata.get(
                "experiment_id",
                "N/A"
            )

            material = metadata.get(
                "material",
                "N/A"
            )

            print(
                f"Rank {result.get('hybrid_rank', 'N/A'):>2} | "
                f"Chunk {result.get('chunk_id', 'N/A'):>4} | "
                f"Experiment {experiment_id:<10} | "
                f"Material: {material}"
            )

            print(
                f"    RRF: "
                f"{result.get('rrf_score', 0):.6f}"
            )

        # ====================================================
        # CROSS-ENCODER
        # ====================================================

        reranked_results = reranker.rerank(
            query=query,
            candidates=enriched_results,
            top_k=5
        )

        # ====================================================
        # RERANKER RESULTS
        # ====================================================

        print("\n" + "-" * 90)
        print("CROSS-ENCODER TOP 5")
        print("-" * 90)

        for result in reranked_results:

            metadata = result.get(
                "metadata",
                {}
            )

            experiment_id = metadata.get(
                "experiment_id",
                "N/A"
            )

            material = metadata.get(
                "material",
                "N/A"
            )

            print(
                f"Rank {result.get('reranker_rank', 'N/A'):>2} | "
                f"Chunk {result.get('chunk_id', 'N/A'):>4} | "
                f"Experiment {experiment_id:<10} | "
                f"Material: {material}"
            )

            print(
                f"    Score: "
                f"{result.get('reranker_score', 0):.4f} | "
                f"Hybrid Rank: "
                f"{result.get('hybrid_rank', 'N/A')}"
            )

        # ====================================================
        # TOP EVIDENCE
        # ====================================================

        print("\n" + "-" * 90)
        print("TOP RERANKED EVIDENCE")
        print("-" * 90)

        for rank, result in enumerate(
            reranked_results[:3],
            start=1
        ):

            metadata = result.get(
                "metadata",
                {}
            )

            experiment_id = metadata.get(
                "experiment_id",
                "N/A"
            )

            text = result.get(
                "text",
                ""
            )

            print(
                f"\n[{rank}] "
                f"Experiment {experiment_id} "
                f"(Chunk {result.get('chunk_id')})"
            )

            print(
                text[:700]
                .replace("\n", " ")
            )

            if len(text) > 700:
                print("...")

        # ====================================================
        # SAVE RESULTS
        # ====================================================

        all_results.append({
            "question_number": question_number,
            "query": query,
            "hybrid_results": enriched_results,
            "reranked_results": reranked_results
        })

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print("\n\n")

    print("=" * 90)
    print("30-QUESTION TEST COMPLETE")
    print("=" * 90)

    print(
        f"\nSuccessfully tested "
        f"{len(all_results)}/"
        f"{len(TEST_QUERIES)} questions."
    )

    return all_results

In [15]:
results_30 = test_30_questions()

30-QUESTION RETRIEVAL TEST

Total questions: 30


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.
Loaded 1200 chunks.



QUESTION 1/30

QUERY:
How does oxygen concentration affect flame behavior in microgravity?

------------------------------------------------------------------------------------------
HYBRID TOP 5
------------------------------------------------------------------------------------------
Rank  1 | Chunk  243 | Experiment EXP-0061   | Material: Nomex
    RRF: 0.029727
Rank  2 | Chunk   15 | Experiment EXP-0004   | Material: Teflon
    RRF: 0.016393
Rank  3 | Chunk  175 | Experiment EXP-0044   | Material: Heptane droplet
    RRF: 0.016129
Rank  4 | Chunk   31 | Experiment EXP-0008   | Material: Nomex
    RRF: 0.016129
Rank  5 | Chunk  289 | Experiment EXP-0073   | Material: Heptane droplet
    RRF: 0.015873

CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...

RERANKING COMPLETE
Top K: 5

--------------------------------------------------
R